# ⚙️ Generative AI for Lightweight Mechanical Bracket Design Using a VAE

**Question:** Can a VAE learn existing mechanical shapes and generate lighter design alternatives automatically?

> Generated silhouettes are educational candidates, not validated components.

👉 **Open the interactive companion:** [https://generative-bracket-vae.streamlit.app](https://generative-bracket-vae.streamlit.app/?stage=start)

## Complete workflow

Bracket masks → encoder → 16-variable latent distribution → decoder → new masks → geometry checks → material-area ranking.

## Interactive learning journey

- [The Lightweight Bracket Brief](https://generative-bracket-vae.streamlit.app/?stage=brief) — Generative Design Objective
- [Bracket Geometry as a Grid](https://generative-bracket-vae.streamlit.app/?stage=pixels) — Binary Occupancy Image
- [A Family of Existing Brackets](https://generative-bracket-vae.streamlit.app/?stage=dataset) — Training Dataset
- [Summarising Each Design](https://generative-bracket-vae.streamlit.app/?stage=encoder) — Encoder
- [A Continuous Design Space](https://generative-bracket-vae.streamlit.app/?stage=latent) — Variational Latent Representation
- [Rebuilding the Bracket](https://generative-bracket-vae.streamlit.app/?stage=decoder) — Decoder
- [Creating New Alternatives](https://generative-bracket-vae.streamlit.app/?stage=generate) — Latent Sampling
- [Checking the Load Path](https://generative-bracket-vae.streamlit.app/?stage=screen) — Geometry Validation
- [Selecting a Lighter Candidate](https://generative-bracket-vae.streamlit.app/?stage=rank) — Material-Area Ranking

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import ndimage
import tensorflow as tf
from tensorflow.keras import layers,Model
SEED=42;np.random.seed(SEED);tf.random.set_seed(SEED)
SIZE,LATENT_DIM=64,16
Y,X=np.mgrid[:SIZE,:SIZE];MOUNTS=[(18,46),(46,18)]

---
# 1. The Lightweight Bracket Brief
### Phase 1 of 6 · The Design Brief

## Part 1 · In mechanical design
A mechanical bracket transfers load between fixed mounting locations. Its holes and support paths must remain while excess material is a candidate for removal.

## Part 2 · The engineering challenge
Drawing only one lighter shape explores little of the design space, while removing material without a load path can produce an unusable component.

## Part 3 · Where the AI comes in
Learn common bracket geometry from examples, generate alternatives, and screen them before comparing material use.

**Mechanical Design:** The Lightweight Bracket Brief → **AI:** Generative Design Objective → `preserve mounts; reduce material`

> 🎬 **See this illustrated and interactive:** [https://generative-bracket-vae.streamlit.app/?stage=brief](https://generative-bracket-vae.streamlit.app/?stage=brief)

## Part 4 · The technical explanation

In [ ]:
def make_bracket(seed=0):
 rng=np.random.default_rng(seed);m=np.zeros((SIZE,SIZE),bool);r=rng.uniform(7,10)
 for cx,cy in MOUNTS:m|=(X-cx)**2+(Y-cy)**2<=r*r
 slope=rng.uniform(-1.12,-.86);intercept=rng.uniform(59,67);thick=rng.uniform(5.5,9)
 m|=(np.abs(Y-(slope*X+intercept))<thick)&(X>10)&(X<54)
 m|=(Y>rng.uniform(42,48))&(Y<rng.uniform(53,58))&(X>9)&(X<35)
 m|=(X>rng.uniform(39,44))&(X<rng.uniform(51,56))&(Y>10)&(Y<39)
 for cx,cy in MOUNTS:m[(X-cx)**2+(Y-cy)**2<rng.uniform(3.5,4.8)**2]=0
 cx,cy=rng.uniform(28,36),rng.uniform(29,39);rx,ry=rng.uniform(4,9),rng.uniform(3,7)
 m[((X-cx)/rx)**2+((Y-cy)/ry)**2<1]=0
 return m.astype("float32")
baseline=make_bracket(4)
plt.imshow(baseline,cmap="gray");plt.title("Baseline bracket");plt.axis("off");plt.show()

## Part 5 · What you just built

**In the notebook:** Define the baseline bracket, fixed mounting zones, and educational screening rules.

**Takeaway:** Generation proposes geometry; engineering checks decide whether it deserves consideration.

[Project overview](https://generative-bracket-vae.streamlit.app/?stage=start) &nbsp;|&nbsp; [Next: Bracket Geometry as a Grid](https://generative-bracket-vae.streamlit.app/?stage=pixels) ▶

---
# 2. Bracket Geometry as a Grid
### Phase 2 of 6 · Geometry as Data

## Part 1 · In mechanical design
A silhouette records where material exists in a two-dimensional bracket profile.

## Part 2 · The engineering challenge
CAD solids are too complex for a first generative notebook, but geometry must still be represented consistently across designs.

## Part 3 · Where the AI comes in
Rasterize every profile onto a 64×64 grid: white pixels are material and black pixels are empty space.

**Mechanical Design:** Bracket Geometry as a Grid → **AI:** Binary Occupancy Image → `64×64: 1 material, 0 empty`

> 🎬 **See this illustrated and interactive:** [https://generative-bracket-vae.streamlit.app/?stage=pixels](https://generative-bracket-vae.streamlit.app/?stage=pixels)

## Part 4 · The technical explanation

In [ ]:
print("Grid:",baseline.shape,"Material pixels:",int(baseline.sum()))
plt.figure(figsize=(7,7));plt.imshow(baseline,cmap="gray",interpolation="nearest");plt.xticks(range(0,65,8));plt.yticks(range(0,65,8));plt.grid(alpha=.25);plt.show()

## Part 5 · What you just built

**In the notebook:** Create and display binary bracket masks with fixed holes.

**Takeaway:** An occupancy grid turns geometry into data while keeping the design visually interpretable.

◀ [Previous: The Lightweight Bracket Brief](https://generative-bracket-vae.streamlit.app/?stage=brief) &nbsp;|&nbsp; [Project overview](https://generative-bracket-vae.streamlit.app/?stage=start) &nbsp;|&nbsp; [Next: A Family of Existing Brackets](https://generative-bracket-vae.streamlit.app/?stage=dataset) ▶

---
# 3. A Family of Existing Brackets
### Phase 2 of 6 · Geometry as Data

## Part 1 · In mechanical design
A design archive contains related brackets with different webs, outer envelopes, lightening holes, and support ribs.

## Part 2 · The engineering challenge
A VAE cannot learn a design language from one component; it needs enough valid variation to distinguish essential structure from optional material.

## Part 3 · Where the AI comes in
Generate hundreds of connected bracket silhouettes while keeping the same mounting zones and varying noncritical geometry.

**Mechanical Design:** A Family of Existing Brackets → **AI:** Training Dataset → `procedural variations of width, curvature, holes`

> 🎬 **See this illustrated and interactive:** [https://generative-bracket-vae.streamlit.app/?stage=dataset](https://generative-bracket-vae.streamlit.app/?stage=dataset)

## Part 4 · The technical explanation

In [ ]:
images=np.array([make_bracket(i) for i in range(900)])[...,None]
rng=np.random.default_rng(SEED);rng.shuffle(images);train,val,test=images[:700],images[700:800],images[800:]
fig,ax=plt.subplots(2,6,figsize=(12,4))
for a,img in zip(ax.ravel(),train[:12]):a.imshow(img[:,:,0],cmap="gray");a.axis("off")
plt.tight_layout();plt.show();print(train.shape,val.shape,test.shape)

## Part 5 · What you just built

**In the notebook:** Build a reproducible synthetic dataset of valid 64×64 masks.

**Takeaway:** The training set defines the design family the VAE can plausibly explore.

◀ [Previous: Bracket Geometry as a Grid](https://generative-bracket-vae.streamlit.app/?stage=pixels) &nbsp;|&nbsp; [Project overview](https://generative-bracket-vae.streamlit.app/?stage=start) &nbsp;|&nbsp; [Next: Summarising Each Design](https://generative-bracket-vae.streamlit.app/?stage=encoder) ▶

---
# 4. Summarising Each Design
### Phase 3 of 6 · Learning a Design Language

## Part 1 · In mechanical design
Engineers describe a bracket through a few characteristics such as web thickness, hole size, curvature, and support distribution.

## Part 2 · The engineering challenge
The image contains 4096 pixels, most of which are redundant descriptions of the same geometric choices.

## Part 3 · Where the AI comes in
Convolutional layers compress the silhouette into parameters of a small latent distribution.

**Mechanical Design:** Summarising Each Design → **AI:** Encoder → `4096 pixels -> mean and log variance`

> 🎬 **See this illustrated and interactive:** [https://generative-bracket-vae.streamlit.app/?stage=encoder](https://generative-bracket-vae.streamlit.app/?stage=encoder)

## Part 4 · The technical explanation

In [ ]:
inp=layers.Input((64,64,1));x=layers.Conv2D(16,3,2,padding="same",activation="relu")(inp);x=layers.Conv2D(32,3,2,padding="same",activation="relu")(x);x=layers.Conv2D(64,3,2,padding="same",activation="relu")(x);x=layers.Flatten()(x);mean=layers.Dense(LATENT_DIM)(x);logvar=layers.Dense(LATENT_DIM)(x)
class Sampling(layers.Layer):
 def call(self,v):
  m,l=v;return m+tf.exp(.5*l)*tf.random.normal(tf.shape(m))
z=Sampling()([mean,logvar]);encoder=Model(inp,[mean,logvar,z]);encoder.summary()

## Part 5 · What you just built

**In the notebook:** Build the convolutional encoder and inspect its mean and log-variance outputs.

**Takeaway:** The encoder converts a detailed shape into a compact design description.

◀ [Previous: A Family of Existing Brackets](https://generative-bracket-vae.streamlit.app/?stage=dataset) &nbsp;|&nbsp; [Project overview](https://generative-bracket-vae.streamlit.app/?stage=start) &nbsp;|&nbsp; [Next: A Continuous Design Space](https://generative-bracket-vae.streamlit.app/?stage=latent) ▶

---
# 5. A Continuous Design Space
### Phase 3 of 6 · Learning a Design Language

## Part 1 · In mechanical design
Useful design parameters change smoothly: a web can widen, a hole can move, or an outer curve can become more pronounced.

## Part 2 · The engineering challenge
A normal autoencoder may leave gaps between known designs, so arbitrary latent points can decode unpredictably.

## Part 3 · Where the AI comes in
The VAE regularises encodings toward a continuous distribution and samples with the reparameterisation trick.

**Mechanical Design:** A Continuous Design Space → **AI:** Variational Latent Representation → `z = μ + σ·ε`

> 🎬 **See this illustrated and interactive:** [https://generative-bracket-vae.streamlit.app/?stage=latent](https://generative-bracket-vae.streamlit.app/?stage=latent)

## Part 4 · The technical explanation

In [ ]:
zin=layers.Input((LATENT_DIM,));d=layers.Dense(8*8*64,activation="relu")(zin);d=layers.Reshape((8,8,64))(d)
for filters in (64,32,16):d=layers.Conv2DTranspose(filters,3,2,padding="same",activation="relu")(d)
decoder=Model(zin,layers.Conv2D(1,3,padding="same",activation="sigmoid")(d))
class VAE(Model):
 def train_step(self,data):
  x=data[0] if isinstance(data,tuple) else data
  with tf.GradientTape() as tape:
   m,l,z=encoder(x);r=decoder(z);rec=tf.reduce_mean(tf.reduce_sum(tf.keras.losses.binary_crossentropy(x,r),axis=(1,2)));kl=-.5*tf.reduce_mean(tf.reduce_sum(1+l-tf.square(m)-tf.exp(l),axis=1));loss=rec+kl
  g=tape.gradient(loss,self.trainable_weights);self.optimizer.apply_gradients(zip(g,self.trainable_weights));return {"loss":loss,"reconstruction":rec,"kl":kl}
vae=VAE();vae.encoder=encoder;vae.decoder=decoder;vae.compile(optimizer="adam")

## Part 5 · What you just built

**In the notebook:** Use a 16-variable latent vector and combine reconstruction and KL losses.

**Takeaway:** A structured latent space makes interpolation and sampling meaningful.

◀ [Previous: Summarising Each Design](https://generative-bracket-vae.streamlit.app/?stage=encoder) &nbsp;|&nbsp; [Project overview](https://generative-bracket-vae.streamlit.app/?stage=start) &nbsp;|&nbsp; [Next: Rebuilding the Bracket](https://generative-bracket-vae.streamlit.app/?stage=decoder) ▶

---
# 6. Rebuilding the Bracket
### Phase 3 of 6 · Learning a Design Language

## Part 1 · In mechanical design
A useful compressed description must reconstruct the material boundary, mounting zones, and support paths.

## Part 2 · The engineering challenge
Too much compression blurs holes or breaks thin webs; too little regularisation merely memorises training images.

## Part 3 · Where the AI comes in
Transpose-convolution layers expand the sampled vector into a pixelwise material probability map.

**Mechanical Design:** Rebuilding the Bracket → **AI:** Decoder → `latent vector -> 64×64 probability map`

> 🎬 **See this illustrated and interactive:** [https://generative-bracket-vae.streamlit.app/?stage=decoder](https://generative-bracket-vae.streamlit.app/?stage=decoder)

## Part 4 · The technical explanation

In [ ]:
vae.fit(train,epochs=25,batch_size=32,verbose=0)
m,_,_=encoder.predict(test[:8],verbose=0);recon=decoder.predict(m,verbose=0)
fig,ax=plt.subplots(2,8,figsize=(15,4))
for i in range(8):ax[0,i].imshow(test[i,:,:,0],cmap="gray");ax[1,i].imshow(recon[i,:,:,0],cmap="gray");ax[0,i].axis("off");ax[1,i].axis("off")
plt.tight_layout();plt.show()

## Part 5 · What you just built

**In the notebook:** Train the VAE and compare original masks with reconstructions.

**Takeaway:** Reconstruction quality shows what geometric information the latent representation retained.

◀ [Previous: A Continuous Design Space](https://generative-bracket-vae.streamlit.app/?stage=latent) &nbsp;|&nbsp; [Project overview](https://generative-bracket-vae.streamlit.app/?stage=start) &nbsp;|&nbsp; [Next: Creating New Alternatives](https://generative-bracket-vae.streamlit.app/?stage=generate) ▶

---
# 7. Creating New Alternatives
### Phase 4 of 6 · Exploring Alternatives

## Part 1 · In mechanical design
Design exploration requires alternatives not copied directly from the archive.

## Part 2 · The engineering challenge
A generated probability image is not yet a manufacturable profile and can change drastically with threshold choice.

## Part 3 · Where the AI comes in
Sample latent vectors, decode them, apply a declared binary threshold, and preserve the unthresholded probability for review.

**Mechanical Design:** Creating New Alternatives → **AI:** Latent Sampling → `sample z -> decoder -> threshold`

> 🎬 **See this illustrated and interactive:** [https://generative-bracket-vae.streamlit.app/?stage=generate](https://generative-bracket-vae.streamlit.app/?stage=generate)

## Part 4 · The technical explanation

In [ ]:
latent=np.random.normal(size=(12,LATENT_DIM)).astype("float32");probabilities=decoder.predict(latent,verbose=0);generated=(probabilities>.5).astype("uint8")
fig,ax=plt.subplots(3,4,figsize=(8,8))
for i,a in enumerate(ax.ravel()):a.imshow(generated[i,:,:,0],cmap="gray");a.set_title(f"Design {chr(65+i)}");a.axis("off")
plt.tight_layout();plt.show()

## Part 5 · What you just built

**In the notebook:** Generate a gallery of previously unseen bracket silhouettes.

**Takeaway:** The VAE generates candidates, not certified components.

◀ [Previous: Rebuilding the Bracket](https://generative-bracket-vae.streamlit.app/?stage=decoder) &nbsp;|&nbsp; [Project overview](https://generative-bracket-vae.streamlit.app/?stage=start) &nbsp;|&nbsp; [Next: Checking the Load Path](https://generative-bracket-vae.streamlit.app/?stage=screen) ▶

---
# 8. Checking the Load Path
### Phase 5 of 6 · Engineering Screening

## Part 1 · In mechanical design
Both mounting regions must remain connected by material for the bracket to have even a plausible load path.

## Part 2 · The engineering challenge
Pixel count alone rewards empty or disconnected images, which appear light only because they have ceased to function as brackets.

## Part 3 · Where the AI comes in
Reject candidates that lose mount coverage or whose required regions are not part of one connected material component.

**Mechanical Design:** Checking the Load Path → **AI:** Geometry Validation → `mount coverage + connected-component test`

> 🎬 **See this illustrated and interactive:** [https://generative-bracket-vae.streamlit.app/?stage=screen](https://generative-bracket-vae.streamlit.app/?stage=screen)

## Part 4 · The technical explanation

In [ ]:
def valid_geometry(mask):
 labels,n=ndimage.label(mask.astype(bool))
 if n==0:return False,"no material"
 mount=[]
 for cx,cy in MOUNTS:
  ring=((X-cx)**2+(Y-cy)**2>=25)&((X-cx)**2+(Y-cy)**2<=81);vals=labels[ring&(mask>0)]
  if len(vals)<18:return False,"mount missing"
  mount.append(np.bincount(vals).argmax())
 if mount[0]!=mount[1]:return False,"mounts disconnected"
 return True,"passes simplified checks"
for i,g in enumerate(generated):print(chr(65+i),valid_geometry(g[:,:,0]))

## Part 5 · What you just built

**In the notebook:** Run connected-component and fixed-mount checks on each generated mask.

**Takeaway:** Validity constraints must come before lightweight ranking.

◀ [Previous: Creating New Alternatives](https://generative-bracket-vae.streamlit.app/?stage=generate) &nbsp;|&nbsp; [Project overview](https://generative-bracket-vae.streamlit.app/?stage=start) &nbsp;|&nbsp; [Next: Selecting a Lighter Candidate](https://generative-bracket-vae.streamlit.app/?stage=rank) ▶

---
# 9. Selecting a Lighter Candidate
### Phase 6 of 6 · Design Review

## Part 1 · In mechanical design
For equal material and thickness, profile area is proportional to mass, making pixel count a simple first comparison.

## Part 2 · The engineering challenge
Area is not stress, stiffness, buckling resistance, fatigue life, manufacturability, or safety factor.

## Part 3 · Where the AI comes in
Rank only valid candidates by material pixels and report reduction relative to the baseline as a simulation result.

**Mechanical Design:** Selecting a Lighter Candidate → **AI:** Material-Area Ranking → `mass proxy = material pixels`

> 🎬 **See this illustrated and interactive:** [https://generative-bracket-vae.streamlit.app/?stage=rank](https://generative-bracket-vae.streamlit.app/?stage=rank)

## Part 4 · The technical explanation

In [ ]:
base=int(baseline.sum());rows=[]
for i,g in enumerate(generated):
 mask=g[:,:,0];valid,reason=valid_geometry(mask);area=int(mask.sum());rows.append(dict(Design=chr(65+i),Material_area=area,Relative_mass=100*area/base,Valid=valid,Reason=reason))
results=pd.DataFrame(rows);display(results)
valid=results[results.Valid].sort_values("Material_area")
if len(valid):
 best=valid.iloc[0];print(f"Selected Design {best.Design}: {100-best.Relative_mass:.1f}% less material by pixel proxy.")
print("Next: CAD reconstruction, loads, FEA, fatigue/buckling/manufacturing review, prototype, and test.")

## Part 5 · What you just built

**In the notebook:** Create the candidate table, select the lightest valid mask, and state the limits of the proxy.

**Takeaway:** A lighter valid silhouette is a candidate for analysis—not an approved engineering design.

◀ [Previous: Checking the Load Path](https://generative-bracket-vae.streamlit.app/?stage=screen) &nbsp;|&nbsp; [Project overview](https://generative-bracket-vae.streamlit.app/?stage=start)

---
# Final engineering conclusion

The VAE samples a continuous latent space to create bracket candidates. Fixed-mount and connectivity checks come before material ranking. Every surviving silhouette still requires proper mechanical analysis and verification.